In [ ]:
# 警告メッセージを無視するためにwarningsモジュールを使用
import warnings, requests, zipfile, io

# 警告メッセージを無視するフィルターを設定
warnings.simplefilter('ignore')

# データ処理のためにpandasモジュールをインポート
import pandas as pd

In [ ]:
# boto3を使ってAWSのサービスにアクセスするためのライブラリをインポート
import boto3

## ① ノートブックを作成し、read_csvメソッドを利用してCSVファイルを読み込む

In [ ]:
# pandasライブラリを使って、AWS S3上のCSVファイルを読み込む
# 事前に必要なデータをS3にアップロードをして、URIをメモしておいてください。
df = pd.read_csv('s3://自分のS3のurl/penguins_binary_classification.csv')

df.head()

## ② 読み込んだデータセットに対して、shapeメソッド、columnsメソッド、dtypesメソッドを実行する

In [ ]:
# DataFrameの行数と列数を確認するためにshape属性を使用
df.shape

In [ ]:
# DataFrameの列名を取得するためにcolumns属性を使用
df.columns

In [ ]:
# DataFrameの各列のデータ型を確認するためにdtypes属性を使用
df.dtypes

## ③ species、islandに対してエンコーディングをする
- `species`: Adelie → 0、Gentoo → 1
- `island`: Biscoe → 0、Dream → 1、Torgersen → 2

In [ ]:
# 変換用の辞書を作る
species_mapper = {'Adelie': 0, 'Gentoo': 1}

# DataFrameから特定の列を取り出し、変換用の辞書で変換
# 変換結果は'species_encoded'という列名でデータフレームに追加
df['species_encoded'] = df['species'].replace(species_mapper)

df.head()

In [ ]:
# 変換用の辞書を作る
island_mapper = {'Biscoe': 0, 'Dream': 1, 'Torgersen': 2}

# DataFrameから特定の列を取り出し、変換用の辞書で変換
# 変換結果は'island_encoded'という列名でデータフレームに追加
df['island_encoded'] = df['island'].replace(island_mapper)

df.head()

In [ ]:
# XGBoostの場合は先頭に目的変数(正解ラベル)、それ以降を特徴量とする
df_for_use = df[['species_encoded',
                 'island_encoded',
                 'bill_length_mm',
                 'bill_depth_mm',
                 'flipper_length_mm',
                 'body_mass_g',
                 'year']].copy()

# モデルの学習で使うデータの先頭5行を表示
df_for_use.head()

## ④ データセットの分割（トレーニング8割・テスト1割・検証1割）

In [ ]:
# 機械学習用ライブラリのsklearnのmode_selectionモジュールからデータ分割関数のtrain_test_split関数をインポート
from sklearn.model_selection import train_test_split

In [ ]:
# 分割1回目
# データセットを トレーニング用(train) と テスト＆検証用(test_and_validate) に分割する
# test_size:テスト＆検証用の割合 検証用1割とテスト用1割の合計なので2割(0.2)を指定
# random_state:乱数シード 任意の値でOK
# stratify:データフレームdf_for_useの目的変数(正解ラベル)の列のばらつき具合を維持して分割する
train, test_and_validate = train_test_split(df_for_use, test_size=0.2, random_state=42, stratify=df['species_encoded'])

In [ ]:
# うまく分割できているか確認
# トレーニングデータセットの行数と列数を表示
print(train.shape)
# トレーニングデータセットの正解ラベルのデータの種類と件数を表示
print(train['species_encoded'].value_counts())

# テスト＆検証用データセットの行数と列数を表示
print(test_and_validate.shape)
# テスト＆検証用データセットの正解ラベルのデータの種類と件数を表示
print(test_and_validate['species_encoded'].value_counts())

In [ ]:
# 分割2回目
# テスト&検証用データセットを テスト用(test) と 検証用(validate) に分割する
# test_size:半分ずつなので5割(0.5)を指定
# random_state:乱数シード 任意の値でOK
# stratify:データフレームtest_and_validateの目的変数(正解ラベル)の列のばらつき具合を維持して分割する
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['species_encoded'])

In [ ]:
# うまく分割できているか確認
# テストデータセットの行数と列数を表示
print(test.shape)
# テストデータセットの正解ラベルのデータの種類と件数を表示
print(test['species_encoded'].value_counts())

# 検証用データセットの行数と列数を表示
print(validate.shape)
# 検証用データセットの正解ラベルのデータの種類と件数を表示
print(validate['species_encoded'].value_counts())

In [ ]:
# S3バケットとプレフィックスの設定
# バケット名は必ずSandboxのものを使う
bucket = '自分のS3バケット名'
prefix = 'penguin'

# トレーニングデータ、テストデータ、検証データのファイル名設定
train_file    = 'penguin_train.csv'
test_file     = 'penguin_test.csv'
validate_file = 'penguin_validate.csv'

# osモジュールのインポート(ファイルパスの生成用)
import os

# S3へアクセスする準備
# boto3セッションからS3リソースを取得
s3_resource = boto3.Session().resource('s3')

# S3にCSVファイルをアップロードするための関数を定義
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    # ★ 以下の部分でS3にデータ(dataframeの内容)をアップしている
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
# DataFrameをS3にアップロードする upload_s3_csv関数を使用

# ファイル名:train_fileの値
# フォルダ名:'train'
# アップするデータ:train(DataFrame)の内容
upload_s3_csv(train_file, 'train', train)

# ファイル名:test_fileの値
# フォルダ名:'test'
# アップするデータ:test(DataFrame)の内容
upload_s3_csv(test_file, 'test', test)

# ファイル名:validate_fileの値
# フォルダ名:'validate'
# アップするデータ:validate(DataFrame)の内容
upload_s3_csv(validate_file, 'validate', validate)

## ⑤ XGBoostモデルのトレーニング（fitメソッド）

In [ ]:
# Amazon SageMaker用のモジュールをインポート
import boto3
from sagemaker.image_uris import retrieve

# SageMakerにおいてXGBoostアルゴリズムを利用するためのコンテナイメージを取得
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
# XGBoostのハイパーパラメータを設定するための辞書を作成
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

In [ ]:
# SageMaker SDKを使用して、XGBoostのモデルをトレーニングおよびデプロイするために必要なモジュールをインポート
import sagemaker

# 出力先のS3ロケーションを設定
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

# SageMakerのXGBoostアルゴリズムを使用するためにXGBoost Estimatorを作成
xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

In [ ]:
# SageMakerのトレーニングジョブに使用するトレーニングデータのS3パスとコンテントタイプを指定
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

# SageMakerのトレーニングジョブに使用する検証データのS3パスとコンテントタイプを指定
validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

# トレーニングジョブのデータチャネルにトレーニングデータと検証データを指定
data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
# XGBoostモデルを学習させるためにfitメソッドを使用
# inputsにはデータチャネルを指定し、logsをFalseに設定してログを表示しないようにする
xgb_model.fit(inputs=data_channels, logs=False)

# ホスティングの準備が完了したことを表示
print('ready for hosting!')

## ⑥ 予測の実行（predictメソッド）

In [ ]:
# XGBoostモデルをデプロイしてエンドポイントを作成
xgb_predictor = xgb_model.deploy(initial_instance_count=1,
                serializer = sagemaker.serializers.CSVSerializer(),
                instance_type='ml.m4.xlarge')
# ホスティングが完了したことを表示
print('Hosting completed!')

In [ ]:
# 再確認としてテスト用データ(test)の中身を見てみる
test.head()

# このデータの1行目 かつ species_encoded列より後ろの列を、推論用のデータとして用いたい
# ※推論させるときは特徴量のみ使う。正解ラベルは渡さない。

In [ ]:
# テストデータから1行目（0から始まる行）を取得し、最初の列以降のデータを表示
row = test.iloc[0:1,1:]

# CSV形式の文字列を一時的に格納するStringIOオブジェクトを作成
batch_X_csv_buffer = io.StringIO()

# DataFrameの1行をCSV形式でStringIOに書き込む（ヘッダーなし、インデックスなし）
row.to_csv(batch_X_csv_buffer, header=False, index=False)

# StringIOからCSV形式の文字列を取得
test_row = batch_X_csv_buffer.getvalue()

# 取得したCSV形式の文字列を表示
print(test_row)

# XGBoostモデルを使用してテストデータの行を予測するためにpredictメソッドを使用
xgb_predictor.predict(test_row)

## ⑦ バッチ変換の実行（transformメソッド）

In [ ]:
# ★ バッチ推論用に 全レコードの正解列以外を抽出
batch_X = test.iloc[:,1:]

# 取得したデータの最初の5行を表示
batch_X.head()

In [ ]:
# バッチデータの入力ファイル名を指定
batch_X_file='batch-in.csv'

# ★ 抽出したDataFrameをS3にアップロード
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
# バッチ変換の出力先を指定
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)

# バッチ変換の入力データのパスを指定
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

# XGBoostトランスフォーマーを作成
xgb_transformer = xgb_model.transformer(instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

# ★ バッチ変換(transform)を実行
xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')

# バッチ変換が完了するのを待機
xgb_transformer.wait()

# バッチ変換完了が完了したことを表示
print('Batch transformation completed!')

In [ ]:
# boto3モジュールを使用してS3クライアントを作成
s3 = boto3.client('s3')

# S3バケットからオブジェクトを取得
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))

# バッチ変換の結果として得られた予測結果を読み込む
# ここではCSVデータを読み込み、列名を指定してDataFrameに変換
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),sep=',',names=['target'])

# DataFrameの最初の5行を表示
target_predicted.head(5)

## ⑧ 予測結果の二値変換と比較（閾値: 0.65）

In [ ]:
# 閾値を設定して二値変換する関数を定義
# binary_convert関数では、確率が指定された閾値（ここでは0.65）より大きい場合は1、それ以外は0に変換されます。
def binary_convert(x):
    threshold = 0.65
    if x > threshold:
        return 1
    else:
        return 0

# 'target'列の各要素に二値変換を適用し、結果を変数target_predicted_binaryに代入する(予測結果のリスト)
target_predicted_binary = target_predicted['target'].apply(binary_convert)

# 予測結果を表示 1:Gentoo 0:Adelie
print(target_predicted_binary.head(10))

# テストデータ(正解が分かっている)を表示
test.head(10)